In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/processed/epl_clean.csv")

In [5]:
df["MatchDate"] = pd.to_datetime(df["MatchDate"])

df = df.sort_values("MatchDate").reset_index(drop=True)

In [6]:
home_df = pd.DataFrame({
    "MatchDate": df["MatchDate"],
    "Season": df["Season"],
    "Team": df["HomeTeam"],
    "Opponent": df["AwayTeam"],
    "Venue": "Home",
    "GoalsScored": df["FullTimeHomeGoals"],
    "GoalsConceded": df["FullTimeAwayGoals"]
})

home_df["Points"] = np.where(
    df["FullTimeResult"] == "H",
    3,
    np.where(df["FullTimeResult"] == "D", 1, 0)
)

In [7]:
away_df = pd.DataFrame({
    "MatchDate": df["MatchDate"],
    "Season": df["Season"],
    "Team": df["AwayTeam"],
    "Opponent": df["HomeTeam"],
    "Venue": "Away",
    "GoalsScored": df["FullTimeAwayGoals"],
    "GoalsConceded": df["FullTimeHomeGoals"]
})

away_df["Points"] = np.where(
    df["FullTimeResult"] == "A",
    3,
    np.where(df["FullTimeResult"] == "D", 1, 0)
)

In [8]:
team_history = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

team_history = team_history.sort_values(
    ["Team", "MatchDate"]
).reset_index(drop=True)

In [9]:
team_history.head()

,MatchDate,Season,Team,Opponent,Venue,GoalsScored,GoalsConceded,Points
0,2000-08-19,2000/01,Arsenal,Sunderland,Away,0,1,0
1,2000-08-21,2000/01,Arsenal,Liverpool,Home,2,0,3
2,2000-08-26,2000/01,Arsenal,Charlton,Home,5,3,3
3,2000-09-06,2000/01,Arsenal,Chelsea,Away,2,2,1
4,2000-09-09,2000/01,Arsenal,Bradford,Away,1,1,1


In [10]:
team_history["RecentForm"] = (
    team_history
    .groupby("Team")["Points"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_history["RecentForm"] = team_history["RecentForm"].fillna(0)

In [11]:
team_history[
    team_history["Team"] == "Liverpool"
][["MatchDate", "Points", "RecentForm"]].head(10)

,MatchDate,Points,RecentForm
8650,2000-08-19,3,0.0
8651,2000-08-21,0,3.0
8652,2000-08-26,1,3.0
8653,2000-09-06,3,4.0
8654,2000-09-09,3,7.0
8655,2000-09-17,1,10.0
8656,2000-09-23,1,8.0
8657,2000-10-01,0,9.0
8658,2000-10-15,3,8.0
8659,2000-10-21,3,8.0


# Merge RecentForm back to original datset

In [12]:
home_form = team_history[
    ["MatchDate", "Team", "RecentForm"]
].rename(
    columns={
        "Team": "HomeTeam",
        "RecentForm": "HomeForm"
    }
)

In [13]:
away_form = team_history[
    ["MatchDate", "Team", "RecentForm"]
].rename(
    columns={
        "Team": "AwayTeam",
        "RecentForm": "AwayForm"
    }
)

In [14]:
df = df.merge(
    home_form,
    on=["MatchDate", "HomeTeam"],
    how="left"
)

In [15]:
df = df.merge(
    away_form,
    on=["MatchDate", "AwayTeam"],
    how="left"
)

In [16]:
df[
    [
        "MatchDate",
        "HomeTeam",
        "AwayTeam",
        "HomeForm",
        "AwayForm"
    ]
].head(20)

,MatchDate,HomeTeam,AwayTeam,HomeForm,AwayForm
0,2000-08-19,Charlton,Man City,0.0,0.0
1,2000-08-19,Chelsea,West Ham,0.0,0.0
2,2000-08-19,Coventry,Middlesbrough,0.0,0.0
3,2000-08-19,Derby,Southampton,0.0,0.0
4,2000-08-19,Leeds,Everton,0.0,0.0
5,2000-08-19,Leicester,Aston Villa,0.0,0.0
6,2000-08-19,Liverpool,Bradford,0.0,0.0
7,2000-08-19,Sunderland,Arsenal,0.0,0.0
8,2000-08-19,Tottenham,Ipswich,0.0,0.0
9,2000-08-20,Man United,Newcastle,0.0,0.0


In [17]:
df[df["HomeTeam"] == "Liverpool"][
    [
        "MatchDate",
        "HomeTeam",
        "HomeForm"
    ]
].head(10)

,MatchDate,HomeTeam,HomeForm
6,2000-08-19,Liverpool,0.0
34,2000-09-06,Liverpool,4.0
43,2000-09-09,Liverpool,7.0
67,2000-09-23,Liverpool,8.0
95,2000-10-21,Liverpool,8.0
107,2000-10-29,Liverpool,8.0
128,2000-11-12,Liverpool,9.0
151,2000-12-02,Liverpool,6.0
168,2000-12-10,Liverpool,6.0
183,2000-12-23,Liverpool,6.0


In [18]:
df.to_csv(
    "../data/processed/epl_features_step1.csv",
    index=False
)

# Rolling Goals Scored

In [19]:
team_history["GoalsScoredAvg5"] = (
    team_history
    .groupby("Team")["GoalsScored"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

In [20]:
team_history["GoalsScoredAvg5"] = (
    team_history["GoalsScoredAvg5"]
    .fillna(0)
)

In [21]:
team_history[
    team_history["Team"]=="Liverpool"
][
    [
        "MatchDate",
        "GoalsScored",
        "GoalsScoredAvg5"
    ]
].head(10)

,MatchDate,GoalsScored,GoalsScoredAvg5
8650,2000-08-19,1,0.000000
8651,2000-08-21,0,1.000000
8652,2000-08-26,3,0.500000
8653,2000-09-06,3,1.333333
8654,2000-09-09,3,1.750000
8655,2000-09-17,1,2.000000
8656,2000-09-23,1,2.000000
8657,2000-10-01,0,2.200000
8658,2000-10-15,4,1.600000
8659,2000-10-21,1,1.800000


In [22]:
home_goals = team_history[
    ["MatchDate","Team","GoalsScoredAvg5"]
].rename(
    columns={
        "Team":"HomeTeam",
        "GoalsScoredAvg5":"HomeGoalsAvg5"
    }
)

In [23]:
away_goals = team_history[
    ["MatchDate","Team","GoalsScoredAvg5"]
].rename(
    columns={
        "Team":"AwayTeam",
        "GoalsScoredAvg5":"AwayGoalsAvg5"
    }
)

In [24]:
df = df.merge(
    home_goals,
    on=["MatchDate","HomeTeam"],
    how="left"
)

In [25]:
df = df.merge(
    away_goals,
    on=["MatchDate","AwayTeam"],
    how="left"
)

In [26]:
df[
    [
        "HomeTeam",
        "AwayTeam",
        "HomeGoalsAvg5",
        "AwayGoalsAvg5"
    ]
].head()

,HomeTeam,AwayTeam,HomeGoalsAvg5,AwayGoalsAvg5
0,Charlton,Man City,0.0,0.0
1,Chelsea,West Ham,0.0,0.0
2,Coventry,Middlesbrough,0.0,0.0
3,Derby,Southampton,0.0,0.0
4,Leeds,Everton,0.0,0.0


In [27]:
df[
    df["HomeTeam"] == "Liverpool"
][[
    "MatchDate",
    "HomeTeam",
    "HomeGoalsAvg5"
]].head(10)

,MatchDate,HomeTeam,HomeGoalsAvg5
6,2000-08-19,Liverpool,0.000000
34,2000-09-06,Liverpool,1.333333
43,2000-09-09,Liverpool,1.750000
67,2000-09-23,Liverpool,2.000000
95,2000-10-21,Liverpool,1.800000
107,2000-10-29,Liverpool,1.400000
128,2000-11-12,Liverpool,2.200000
151,2000-12-02,Liverpool,2.400000
168,2000-12-10,Liverpool,2.400000
183,2000-12-23,Liverpool,1.200000


In [28]:
df.to_csv(
    "../data/processed/epl_features_step2.csv",
    index=False
)

# HomeGoalsConcededAvg5
# AwayGoalsConcededAvg5

In [29]:
team_history["GoalsConcededAvg5"] = (
    team_history
    .groupby("Team")["GoalsConceded"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

In [30]:
team_history["GoalsConcededAvg5"] = (
    team_history["GoalsConcededAvg5"]
    .fillna(0)
)

In [31]:
home_conceded = team_history[
    ["MatchDate","Team","GoalsConcededAvg5"]
].rename(
    columns={
        "Team":"HomeTeam",
        "GoalsConcededAvg5":"HomeGoalsConcededAvg5"
    }
)

In [32]:
away_conceded = team_history[
    ["MatchDate","Team","GoalsConcededAvg5"]
].rename(
    columns={
        "Team":"AwayTeam",
        "GoalsConcededAvg5":"AwayGoalsConcededAvg5"
    }
)

In [33]:
df = df.merge(
    home_conceded,
    on=["MatchDate","HomeTeam"],
    how="left"
)

df = df.merge(
    away_conceded,
    on=["MatchDate","AwayTeam"],
    how="left"
)

In [34]:
df[
    [
        "HomeTeam",
        "AwayTeam",
        "HomeGoalsConcededAvg5",
        "AwayGoalsConcededAvg5"
    ]
].head()

,HomeTeam,AwayTeam,HomeGoalsConcededAvg5,AwayGoalsConcededAvg5
0,Charlton,Man City,0.0,0.0
1,Chelsea,West Ham,0.0,0.0
2,Coventry,Middlesbrough,0.0,0.0
3,Derby,Southampton,0.0,0.0
4,Leeds,Everton,0.0,0.0


# Home Strength & Away Strength


In [35]:
team_history["HomeWin"] = np.where(
    (team_history["Venue"] == "Home") &
    (team_history["Points"] == 3),
    1,
    0
)

In [36]:
team_history["HomeWinRate5"] = (
    team_history[team_history["Venue"] == "Home"]
    .groupby("Team")["HomeWin"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

In [37]:
team_history["AwayWin"] = np.where(
    (team_history["Venue"] == "Away") &
    (team_history["Points"] == 3),
    1,
    0
)

In [38]:
team_history["AwayWinRate5"] = (
    team_history[team_history["Venue"] == "Away"]
    .groupby("Team")["AwayWin"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

In [39]:
team_history["HomeWinRate5"] = team_history["HomeWinRate5"].fillna(0)
team_history["AwayWinRate5"] = team_history["AwayWinRate5"].fillna(0)

In [40]:
home_strength = team_history[
    ["MatchDate", "Team", "HomeWinRate5"]
].rename(
    columns={
        "Team": "HomeTeam",
        "HomeWinRate5": "HomeWinRate5"
    }
)

df = df.merge(
    home_strength,
    on=["MatchDate", "HomeTeam"],
    how="left"
)

In [41]:
away_strength = team_history[
    ["MatchDate", "Team", "AwayWinRate5"]
].rename(
    columns={
        "Team": "AwayTeam",
        "AwayWinRate5": "AwayWinRate5"
    }
)

df = df.merge(
    away_strength,
    on=["MatchDate", "AwayTeam"],
    how="left"
)

In [42]:
df[
    [
        "HomeTeam",
        "AwayTeam",
        "HomeWinRate5",
        "AwayWinRate5"
    ]
].head(10)

,HomeTeam,AwayTeam,HomeWinRate5,AwayWinRate5
0,Charlton,Man City,0.0,0.0
1,Chelsea,West Ham,0.0,0.0
2,Coventry,Middlesbrough,0.0,0.0
3,Derby,Southampton,0.0,0.0
4,Leeds,Everton,0.0,0.0
5,Leicester,Aston Villa,0.0,0.0
6,Liverpool,Bradford,0.0,0.0
7,Sunderland,Arsenal,0.0,0.0
8,Tottenham,Ipswich,0.0,0.0
9,Man United,Newcastle,0.0,0.0
